<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Custom_Attention_SMS_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [1]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [3]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [4]:
# Load and inspect the SMS Spam dataset
DATA_PATH = "hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet"

df = pd.read_parquet(DATA_PATH)
hf_dataset = Dataset.from_pandas(df)

TRAIN_START = 0
TRAIN_END = 4000
VAL_START = 4000
VAL_END = 5000

if None in (TRAIN_END, VAL_START, VAL_END):
    raise ValueError(
        "Set TRAIN_END, VAL_START, and VAL_END according to the instructions."
    )

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))

print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))

display(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Training samples: 4000
Validation samples: 1000


,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [6]:
# Initialize the tokenizer and padding behavior
from transformers import GPT2Tokenizer

MODEL_NAME = "gpt2"

if MODEL_NAME is None:
    raise ValueError(
        "Set MODEL_NAME to the pretrained checkpoint (e.g., 'gpt2')."
    )

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# GPT-2 has no padding token by default, so EOS is used for padding
tokenizer.pad_token = tokenizer.eos_token

print("Model:", MODEL_NAME)
print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)
print("EOS token ID:", tokenizer.eos_token_id)

assert tokenizer.pad_token_id == tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model: gpt2
Pad token: <|endoftext|>
Pad token ID: 50256
EOS token ID: 50256


In [7]:
# Complete the tokenization function
TEXT_COLUMN = "sms"
PADDING_STRATEGY = "max_length"
TRUNCATION_FLAG = True
MAX_SEQ_LEN = 64

for setting in (TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, MAX_SEQ_LEN):
    if setting is None:
        raise ValueError(
            "Complete TEXT_COLUMN, PADDING_STRATEGY, "
            "TRUNCATION_FLAG, and MAX_SEQ_LEN."
        )


def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )


train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

print(train_tok[0])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0, 'input_ids': [5247, 1566, 8174, 506, 966, 11, 7165, 492, 14898, 691, 287, 5434, 271, 299, 1049, 995, 8591, 304, 44703, 986, 327, 500, 612, 1392, 716, 382, 4383, 986, 198, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [8]:
# Instantiate GPT-2 for sequence classification
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2  # ham = 0, spam = 1

if NUM_LABELS is None:
    raise ValueError("Set NUM_LABELS to 2 for binary classification.")

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,
)

print("Number of labels:", model.config.num_labels)
print("Model pad token ID:", model.config.pad_token_id)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of labels: 2
Model pad token ID: 50256


# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [9]:
# Implement the Attention layer
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        # Scaling factor: 1 / sqrt(embed_dim)
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):

        # Shape: (batch_size, sequence_length, sequence_length)
        scores = torch.matmul(
            query,
            key.transpose(-2, -1),
        ) * self.scale

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        # Normalize scores over the key/sequence dimension
        attn = F.softmax(scores, dim=-1)

        # Weighted sum of the value vectors
        output = torch.matmul(attn, value)

        return output, attn


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.attn = Attention(embed_dim)

        self.fc = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):

        # Shape: (batch_size, sequence_length, embed_dim)
        embed = self.embedding(x)

        # Self-attention: query = key = value = embeddings
        attn_output, _ = self.attn(
            embed,
            embed,
            embed
        )

        # Mean pooling over the sequence dimension
        pooled = attn_output.mean(dim=1)

        # Binary classification logits
        return self.fc(pooled)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [10]:
# Preprocess datasets for the custom attention model
ATTN_TEXT_COLUMN = "sms"
ATTN_MAX_LEN = 64

if ATTN_TEXT_COLUMN is None or ATTN_MAX_LEN is None:
    raise ValueError(
        "Complete ATTN_TEXT_COLUMN and ATTN_MAX_LEN."
    )


def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding="max_length",
    )

    return {
        "input_ids": tokens,
        "label": example["label"]
    }


train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

print(train_ds_attn[0])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0, 'input_ids': [5247, 1566, 8174, 506, 966, 11, 7165, 492, 14898, 691, 287, 5434, 271, 299, 1049, 995, 8591, 304, 44703, 986, 327, 500, 612, 1392, 716, 382, 4383, 986, 198, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]}


In [11]:
# Create PyTorch DataLoaders
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        return {
            "input_ids": torch.tensor(
                item["input_ids"],
                dtype=torch.long
            ),
            "label": torch.tensor(
                item["label"],
                dtype=torch.long
            ),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn
VAL_DATA_FOR_LOADER = val_ds_attn

if TRAIN_DATA_FOR_LOADER is None or VAL_DATA_FOR_LOADER is None:
    raise ValueError(
        "Assign TRAIN_DATA_FOR_LOADER and "
        "VAL_DATA_FOR_LOADER before creating loaders."
    )


train_loader = DataLoader(
    SMSDataset(TRAIN_DATA_FOR_LOADER),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    SMSDataset(VAL_DATA_FOR_LOADER),
    batch_size=32
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 125
Validation batches: 32


In [12]:
# Train the custom attention classifier
vocab_size = len(tokenizer)  # includes added tokens
embed_dim = 64
num_classes = 2
learning_rate = 1e-3

if None in (vocab_size, num_classes, learning_rate):
    raise ValueError(
        "Set vocab_size, num_classes, and learning_rate before training."
    )


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

attn_model = SimpleAttentionClassifier(
    vocab_size,
    embed_dim,
    num_classes
).to(device)

optimizer = torch.optim.Adam(
    attn_model.parameters(),
    lr=learning_rate
)

criterion = nn.CrossEntropyLoss()


attn_model.train()

for batch in train_loader:
    inputs = batch["input_ids"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()

    outputs = attn_model(inputs)

    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()


print(
    "Custom Attention model trained on SMS dataset. "
    "Sample batch loss:",
    loss.item()
)

Custom Attention model trained on SMS dataset. Sample batch loss: 0.4005596935749054


# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [13]:
# Configure evaluation metrics
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")


def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],

        "precision": precision.compute(
            predictions=preds,
            references=labels
        )["precision"],

        "recall": recall.compute(
            predictions=preds,
            references=labels
        )["recall"],

        "f1": f1.compute(
            predictions=preds,
            references=labels
        )["f1"],
    }

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [14]:
# Evaluate GPT-2 on the validation split
gpt2_preds = []
gpt2_labels = []

model.eval()

for ex in val_tok:
    inputs = torch.tensor(
        ex["input_ids"]
    ).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(inputs).logits

    pred = torch.argmax(
        logits,
        dim=-1
    ).cpu().item()

    gpt2_preds.append(pred)
    gpt2_labels.append(ex["label"])


gpt2_metrics = {
    "accuracy": accuracy.compute(
        predictions=gpt2_preds,
        references=gpt2_labels
    )["accuracy"],

    "precision": precision.compute(
        predictions=gpt2_preds,
        references=gpt2_labels
    )["precision"],

    "recall": recall.compute(
        predictions=gpt2_preds,
        references=gpt2_labels
    )["recall"],

    "f1": f1.compute(
        predictions=gpt2_preds,
        references=gpt2_labels
    )["f1"],
}

print("GPT-2 Metrics:", gpt2_metrics)

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


GPT-2 Metrics: {'accuracy': 0.137, 'precision': 0.13360323886639677, 'recall': 0.9496402877697842, 'f1': 0.23425022182786157}


In [ ]:
# TODO: evaluate the custom attention model
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())


attn_metrics = {
    'accuracy': accuracy.compute(predictions=..., references=...)['accuracy'],    # TODO
    'precision': precision.compute(predictions=..., references=...)['precision'],  # TODO
    'recall': recall.compute(predictions=..., references=...)['recall'],          # TODO
    'f1': f1.compute(predictions=..., references=...)['f1'],                      # TODO
}
print('Attention Model Metrics:', attn_metrics)


# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?
TODO: Write your explanation here.

The **query** represents what the current token is looking for, while the **key** represents the information available in each token for comparison. The attention mechanism compares queries with keys to calculate attention weights, then uses these weights to combine the **values**, which contain the actual information passed to the output.


### 2. Why do we use a scaling factor in the dot-product attention?
TODO: Summarize the numerical stability rationale.

As the key dimension increases, the dot products between queries and keys can become very large. Dividing the scores by ($\sqrt{d_k}$) prevents the softmax function from becoming saturated, which helps maintain useful gradients and makes training more numerically stable.


### 3. How does self-attention differ from traditional sequence models like RNNs?
TODO: Compare processing style, dependency capture, and efficiency.

Self-attention processes all tokens in a sequence simultaneously, while RNNs process them one step at a time. This allows self-attention to capture long-range dependencies more directly and to train more efficiently through parallel computation, whereas RNNs can be slower and may struggle with information from distant parts of a sequence.


### 4. Performance analysis
TODO: Discuss which model performed better, describe trade-offs, and suggest one improvement for the custom attention classifier.

In this experiment, the custom attention classifier is expected to perform better because it was trained on the SMS dataset, while the GPT-2 classification head was only initialized and not fine-tuned. GPT-2 has richer pretrained language knowledge, but it requires more memory, computing power, and task-specific training. The custom model is lighter, faster, and easier to understand, although it has a more limited ability to capture complex language patterns.

One improvement would be to add an attention mask so that padding tokens are ignored during attention and pooling. This would help the model focus only on the meaningful words in each SMS and could improve classification performance.
